### [NO] traces during EBCC simulation (Reproduce Suppl. Figure 4)

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import dill
import json
import sys
import os
from scipy import spatial
import matplotlib.pyplot as plt
import pickle
import h5py

In [ ]:
dict_path = 'data/reshaped_ev_points.csv'
ev_points = pd.read_csv(dict_path)
cluster = np.unique(ev_points['cluster'].values)
with open("/home/csartor1/code/NODS/network_configuration.json", "r") as json_file:
    net_config = json.load(json_file)
CS_burst_dur = net_config["devices"]["CS"]["parameters"]["burst_dur"]
CS_start_first = float(net_config["devices"]["CS"]["parameters"]["start_first"])
between_start = net_config["devices"]["CS"]["parameters"]["between_start"]
n_trials = net_config["devices"]["CS"]["parameters"]["n_trials"]
US_start_first = float(net_config["devices"]["US"]["parameters"]["start_first"])
CS_color = net_config["colors"]["CS"]
US_color = net_config["colors"]["US"]

### Set time limits

In [ ]:
rand_clusters = cluster
clusters_df = ev_points[ev_points['cluster'].isin(rand_clusters)]
ev_points_n_clusters = clusters_df[['ev_points_id', 'x', 'y', 'z']].values

t0 = 0
tf = 2000

### Extract PC receiving CS stimuli

In [ ]:
with h5py.File("/home/csartor1/code/nearlab_repo/NODS/data/pf-PC.hdf5", "r") as f:
    id_granule = f['pf_pc_connections'][:]

id_granule
rel_dist_path = '/home/csartor1/code/NODS/data/relative_dist.csv'
# source_id, nos_id, ev_points_id, d, cluster
relative_dist = pd.read_csv(rel_dist_path, header = None)
matched_grc = relative_dist[relative_dist.iloc[:,1].isin(id_granule)]
id_ev = np.unique(matched_grc.iloc[:,3])

In [ ]:
NO_conc_CS = []
NO_median = []
noise_levels = [0, 4, 8]
for noise in noise_levels:    
    result_path = f'results/test/w_NO/{noise}Hz/sim_NO_conc_CS/NO_concentration_data/'

    concentration_CS_clusters = []

    for i in range(t0,tf,5):
        file_path = result_path + f'NO_concentration_t_{i}.csv'
        df_conc = pd.read_csv(file_path,header=None)
        matched_conc = df_conc[df_conc.iloc[:, 1].isin(id_ev)]
        concentrations = matched_conc.iloc[:, 2].values
        concentration_CS_clusters.append(concentrations)

    concentration_data_CS = np.array(concentration_CS_clusters)
    median_CS = np.median(concentration_data_CS,axis=1)

    NO_conc_CS.append(concentration_data_CS)
    NO_median.append(median_CS)

NO_conc_CS = np.array(NO_conc_CS)
NO_median = np.array(NO_median)

In [ ]:
# SET THRESHOLD FOR NO CONCENTRATION

threshold = 100

def sig(x, A=2, B=170, C=5):
    return A / (1 + np.exp(-(x - B) / C))

### Plot median [NO] and $G_{[NO]}$ in for pf-PC synapses receiving CS stimuli

In [ ]:
plt.rcParams.update({'font.size': 12},)
plt.figure(figsize=(12,2.5))
greens = ['#b9fbc0', '#70e000', '#008000', '#004225']
for i,noise in enumerate(noise_levels):
    
    plt.plot(np.arange(t0,tf,5),NO_median[i], linewidth=2, color = greens[i], label=f'{noise}Hz')

plt.xlabel('time [ms]')
plt.ylabel('NO Concentration [pM]')
plt.hlines(y = threshold, xmin = t0, xmax = tf,linestyles = 'dashed', linewidth=2, color = 'black')
plt.vlines([CS_start_first+t0+500*i for i in range(int(((tf-t0))/500))], ymin = 0, ymax = 150, linewidth=2.5, color=CS_color)
plt.vlines([CS_start_first + CS_burst_dur + t0 +500*i for i in range(int(((tf-t0))/500))], ymin = 0, ymax = 150, linewidth=2.5, color=US_color)
#plt.grid(visible=True)
plt.title('NO produced by nNOS', fontsize=16,fontweight ='bold')
plt.yticks([0,25,50,75,100,125,150])
plt.legend(loc = 'lower right')
#plt.savefig('vm_neuron.png')

plt.savefig('NO_trace_noises.png')
plt.show()

In [ ]:
plt.rcParams.update({'font.size': 12},)
plt.figure(figsize=(12,2.5))
for i,noise in enumerate(noise_levels):
    median_GNO_CS= sig(x=NO_median[i], A=1, B=threshold)
    plt.plot(np.arange(t0,tf,5),median_GNO_CS, color = greens[i],linewidth=2, label=f'{noise}Hz')

    plt.xlabel('time [ms]')
    plt.ylabel('$G_{[NO]}$')
    plt.vlines([CS_start_first+t0+500*i for i in range(int(((tf-t0))/500))], ymin = 0, ymax = 1, linewidth=2.5, color=CS_color)
    plt.vlines([CS_start_first + CS_burst_dur + t0 +500*i for i in range(int(((tf-t0))/500))], ymin = 0, ymax = 1, linewidth=2.5, color=US_color)
    #plt.grid(visible=True)
    plt.title('$G_{[NO]}$ during simulation', fontsize=16,fontweight ='bold')
    plt.legend(loc = 'lower right')
    #plt.savefig('vm_neuron.png')

    plt.savefig('GNO_trace_noises.png')
    plt.show()

### Extract and plot [NO] and $G_{[NO]}$ comparing CS+noise and only noise (Inside and Outside the CS stimuli area)

In [ ]:
NO_conc_noise = []
NO_median_noise = []
for noise in noise_levels:    
    result_path = f'results/test/w_NO/{noise}Hz/sim_NO_conc_CS/NO_concentration_data/'

    concentration_noise = []

    for i in range(t0,tf,5):
        file_path = result_path + f'NO_concentration_t_{i}.csv'
        df_conc = pd.read_csv(file_path,header=None)
        matched_conc = df_conc[~df_conc.iloc[:, 1].isin(id_ev)]
        concentrations = matched_conc.iloc[:, 2].values
        concentration_noise.append(concentrations)

    concentration_data_noise = np.array(concentration_noise)
    median_noise = np.median(concentration_data_noise,axis=1)

    NO_conc_noise.append(concentration_data_noise)
    NO_median_noise.append(median_noise)

NO_conc_noise= np.array(NO_conc_noise)
NO_median_noise = np.array(NO_median_noise)

In [ ]:
median_GNO_noise = []
threshold = 100
for i,noise in enumerate(noise_levels):
    median_GNO_noise.append(sig(x=NO_median_noise[i], A=1, B=threshold))

In [ ]:
plt.rcParams.update({'font.size': 16})
fig, axs = plt.subplots(3, 2, figsize=(16, 10), sharex=True, gridspec_kw={'height_ratios': [3, 3, 3]})
fig.tight_layout(pad=3.0)



# Create the plots
for i, noise in enumerate(noise_levels):
    # Left column: NO concentration
    axs[i, 0].plot(np.arange(t0, tf, 5), NO_median[i], linewidth=2.5, color='g', label=f'CS + noise')
    axs[i, 0].plot(np.arange(t0, tf, 5), NO_median_noise[i], linewidth=2.5, color='r', label=f'noise only')
    axs[i, 0].axhline(y=threshold, xmin=0, xmax=1, linestyle='dashed', linewidth=2.5, color='black', label = 'threshold')
    axs[i, 0].set_yticks([0, 25, 50, 75, 100, 125, 150])
    
    # Add noise level text to the left plot
    axs[i, 0].text(0.05, 0.9, f'{noise} Hz', transform=axs[i, 0].transAxes, 
                 fontsize=16, fontweight='bold', bbox=dict(facecolor='white'))
    
    # Add vertical lines for CS and US in left column
    for j in range(int((tf-t0)/500)):
        axs[i, 0].axvline(x=CS_start_first+t0+500*j, ymin=0, ymax=1, linewidth=2.5, color=CS_color, label = 'CS start')
        axs[i, 0].axvline(x=CS_start_first+CS_burst_dur+t0+500*j, ymin=0, ymax=1, linewidth=2.5, color=US_color, label = 'CS end')
    
    if i == 3:  # Only add x-label on the bottom plot
        axs[i, 0].set_xlabel('time [ms]')
    
    # Right column: G[NO]
    median_GNO_CS = sig(x=NO_median[i], A=1, B=threshold)
    median_GNO_noise = sig(x=NO_median_noise[i], A=1, B=threshold)
    axs[i, 1].plot(np.arange(t0, tf, 5), median_GNO_CS, color='g', linewidth=2.5, label='CS + noise')
    axs[i, 1].plot(np.arange(t0, tf, 5), median_GNO_noise, color='r', linewidth=2.5, label='only noise')
    axs[i, 1].set_ylim(-0.05, 1.05)
    axs[i, 1].set_yticks([0, 0.5, 1])
    
    # Add vertical lines for CS and US in right column
    for j in range(int((tf-t0)/500)):
        axs[i, 1].axvline(x=CS_start_first+t0+500*j, ymin=0, ymax=1, linewidth=3, color=CS_color)
        axs[i, 1].axvline(x=CS_start_first+CS_burst_dur+t0+500*j, ymin=0, ymax=1, linewidth=3, color=US_color)
    
    if i == 2:  # Only add x-label on the bottom plot
        axs[i, 1].set_xlabel('time [ms]')
    
# Add bold labels for each column
#fig.text(0.25, 0.02, 'time [ms]', ha='center', fontsize=16, fontweight='bold')
#fig.text(0.75, 0.02, 'time [ms]', ha='center', fontsize=16, fontweight='bold')
fig.text(0.02, 0.5, 'NO concentration [pM]', va='center', fontsize=16, fontweight='bold', rotation=90)
fig.text(0.48, 0.5, 'G[NO]', va='center', fontsize=18, fontweight='bold', rotation=90)

# Add a legend to the figure (outside the plots)
lines1, labels1 = axs[0, 0].get_legend_handles_labels()
#lines2, labels2 = axs[0, 1].get_legend_handles_labels()
fig.legend(lines1[:5], labels1[:5], loc='upper center', ncol=5, bbox_to_anchor=(0.5, 0.98), frameon=True)

# Adjust spacing between subplots
plt.subplots_adjust(hspace=0.4, left=0.08, right=0.92, top=0.9, bottom=0.08)

# Show the plot
plt.show()